In [ ]:
EXP_DIR="../results/framework_validation_0827"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

class CustPalette:
  primary_dark="#262699"
  primary="#2945ad"
  primary_med="#6882db"
  primary_light="#a9b9f0"
  primary_x_light="#d7e0ff"
  secondary="#ff3477"
  secondary_light="#ff84ad"
  secondary_x_light="#ffb4cd"
  tertiary="#26cc90"
  grey_light='#cbd5e1'
  grey_med='#727B8D'
  grey_dark='#17191C'
  green='#5fdf50'

In [ ]:
results_df = pd.read_csv(f"{EXP_DIR}/benchmark_results.csv")
# datasets_df = pd.read_csv(f"{EXP_DIR}/benchmark_datasets.csv")

optuna_mask = results_df['optimiser'] == 'optuna'
llm_mask = results_df['optimiser'] == 'llm'

In [ ]:
results_df.columns

# Success Rates

## AUPRC Target

In [ ]:
# Convergence after Raw Generation
optuna_raw_gen_conv_n = results_df.loc[optuna_mask, 'raw_gen_converged'].sum()
llm_raw_gen_conv_n = results_df.loc[llm_mask, 'raw_gen_converged'].sum()

print(f"OPTUNA: Number of converged raw generations: {optuna_raw_gen_conv_n} ({optuna_raw_gen_conv_n / len(results_df[optuna_mask]) * 100:.2f}%)")
print(f"LLM: Number of converged raw generations: {llm_raw_gen_conv_n} ({llm_raw_gen_conv_n / len(results_df[llm_mask]) * 100:.2f}%)")

In [ ]:
# Final Convergence
results_df['target_auprc_met'] = results_df['target_auprc'] <= results_df['auprc']

unique_auprc_targets = results_df['target_auprc'].unique()
for (optimiser, target), group in results_df.groupby(['optimiser','target_auprc']):
  print(f"\n==== {optimiser.upper()}, TARGET = {target*100}% ====")
  unmet_auprc_target = group[~group['target_auprc_met']]
  missed_n = len(unmet_auprc_target)
  print(f"Missed target in {missed_n} datasets out of {len(group)} ({missed_n / len(group) * 100:.2f}%)")
  auprc_discr = unmet_auprc_target['target_auprc'] - unmet_auprc_target['auprc']
  average_auprc_discr = auprc_discr.mean()
  max_auprc_discr = auprc_discr.max()

  print(f"Average AUPRC discrepancy when target is missed: {average_auprc_discr*100:.2f}%")
  print(f"Maximum AUPRC discrepancy when target is missed: {max_auprc_discr*100:.2f}%")

In [ ]:
results_df["auprc_margin"] = results_df["auprc"] - results_df["target_auprc"]

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(
  data=results_df,
  x="target_auprc",
  y="auprc_margin",
  hue="optimiser",
  palette=[CustPalette.primary, CustPalette.secondary],
  medianprops={'linewidth': 2,'color':'white'}
)

# Reference zero-error line
ax.axhline(0, color=CustPalette.green, linestyle="--", linewidth=1.5, label="Target Floor")

ax.set_xlabel("Target AUPRC")
ax.set_ylabel("Clearance Margin (Achieved AUPRC - Target)")
ax.set_title("AUPRC Target Clearance by Optimiser")
ax.legend()
plt.show()

## Target disparities

In [ ]:
# Bias Application Convergence Rate
optuna_bias_conv = results_df.loc[optuna_mask, "bias_converged"].sum()
llm_bias_conv = results_df.loc[llm_mask, "bias_converged"].sum()

print(f"OPTUNA: Number of bias converged runs: {optuna_bias_conv} ({optuna_bias_conv / len(results_df[optuna_mask]) * 100:.2f}%)")
print(f"LLM: Number of bias converged runs: {llm_bias_conv} ({llm_bias_conv / len(results_df[llm_mask]) * 100:.2f}%)")

In [ ]:
results_df["recall_disp_error"] = (
    results_df["recall_disp"] - results_df["target_recall_disp"]
)
results_df["ppv_disp_error"] = (
    results_df["ppv_disp"] - results_df["target_ppv_disp"]
)

results_df["recall_in_tol"] = (
    results_df["recall_disp_error"].abs() <= results_df["disp_tolerance"]
)
results_df["ppv_in_tol"] = (
    results_df["ppv_disp_error"].abs() <= results_df["disp_tolerance"]
)

for opt in ["llm", "optuna"]:
  sub = results_df[results_df["optimiser"] == opt]
  print(f"\n==== {opt.upper()} DISPARITY TOLERANCE COMPLIANCE ====")

  for (target_disp, target_recall_disp, target_ppv_disp), group in sub.groupby(['target_disp', 'target_recall_disp', 'target_ppv_disp']):
    print(f"\n---TARGET DISPARITY: {target_disp.upper()}---")
    if target_disp == "recall" or target_disp == "both":
      print(f"Target Recall Disparity: {target_recall_disp}")
      print(
        f"Recall Disparity within ±tol: {group['recall_in_tol'].sum()}/{len(group)} ({group['recall_in_tol'].mean()*100:.2f}%)"
      )
    if target_disp == "ppv" or target_disp == "both":
      print(f"Target PPV Disparity: {target_recall_disp}")
      print(
        f"PPV Disparity within ±tol:    {group['ppv_in_tol'].sum()}/{len(group)} ({group['ppv_in_tol'].mean()*100:.2f}%)"
      )

In [ ]:
def build_disparity_table(df):
  records = []
  group_cols = [
      'optimiser',
      'target_disp',
      'target_recall_disp',
      'target_ppv_disp',
  ]

  for (opt, t_disp, t_recall, t_ppv), group in df.groupby(group_cols):
    total = len(group)

    # Recall Disparity Compliance
    if t_disp in ['recall', 'both']:
      r_met = group['recall_in_tol'].sum()
      r_str = f'{r_met}/{total} ({r_met / total * 100:.1f}%)'
    else:
      r_str = '—'
      t_recall = '—'

    # PPV Disparity Compliance
    if t_disp in ['ppv', 'both']:
      p_met = group['ppv_in_tol'].sum()
      p_str = f'{p_met}/{total} ({p_met / total * 100:.1f}%)'
    else:
      p_str = '—'
      t_ppv = '—'

    records.append({
        'Optimiser': opt.upper(),
        'Target Mode': t_disp.upper(),
        'Target Recall': t_recall,
        'Recall In Tol': r_str,
        'Target PPV': t_ppv,
        'PPV In Tol': p_str,
        'Total Runs': total,
    })

  return pd.DataFrame(records)


disp_summary_df = build_disparity_table(results_df)
print(disp_summary_df.to_markdown())

In [ ]:
avg_tol = results_df["disp_tolerance"].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

metrics = [
    (
        "target_recall_disp",
        "recall_disp_error",
        "Recall Disparity Error",
        axes[0],
    ),
    ("target_ppv_disp", "ppv_disp_error", "PPV Disparity Error", axes[1]),
]

for target_col, error_col, title, ax in metrics:
  plot_data = results_df[results_df[target_col] != 0.0]

  sns.boxplot(
      data=plot_data,
      x=target_col,
      y=error_col,
      hue="optimiser",
      palette=[CustPalette.primary, CustPalette.secondary],
      medianprops={"linewidth": 2, "color": "white"},
      ax=ax,
  )

  # Tolerance band & zero reference line
  ax.axhline(
      0, color=CustPalette.grey_dark, linestyle="-", linewidth=1, alpha=0.7
  )
  ax.axhspan(
      -avg_tol,
      avg_tol,
      color=CustPalette.green,
      alpha=0.15,
      label=f"Tolerance (±{avg_tol:.2f})",
  )
  ax.axhline(
      avg_tol,
      color=CustPalette.green,
      linestyle="--",
      linewidth=1.2,
      alpha=0.8,
  )
  ax.axhline(-avg_tol, color=CustPalette.green, linestyle="--", linewidth=1.2)

  ax.set_title(title)
  ax.set_xlabel("Target Disparity")
  ax.set_ylabel("Error (Achieved − Target)")

  # Remove subplot legend
  if ax.get_legend() is not None:
    ax.get_legend().remove()

# Extract unique handles/labels from the first subplot
handles, labels = axes[0].get_legend_handles_labels()

# Place single unified horizontal legend below both plots
fig.legend(
    handles=handles,
    labels=labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=len(labels),
    frameon=False,
)

plt.tight_layout()
plt.show()

---

# Search Efficiency

## Retries summary by phase

In [ ]:
# Summary statistics for trials, retries, and wall-clock time
efficiency_cols = ['raw_gen_trials', 'retry_count', 'wall_clock_sec']

efficiency_summary = (
    results_df.groupby('optimiser')[efficiency_cols]
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(2)
)

efficiency_summary

## Cumulative completion rate by phase

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, PercentFormatter
import seaborn as sns

results_df["bias_trials"] = (
    results_df["retry_count"] - results_df["raw_gen_trials"]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Raw Generation ECDF
sns.ecdfplot(
    data=results_df,
    x="raw_gen_trials",
    hue="optimiser",
    palette=[CustPalette.primary, CustPalette.secondary],
    linewidth=2.5,
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Raw Gen: Cumulative Completion Rate")
axes[0].set_xlabel("Trial Count (Retries)")
axes[0].set_ylabel("Percentage Completed")
axes[0].xaxis.set_major_locator(MultipleLocator(2))

# 2. Bias Phase ECDF
sns.ecdfplot(
    data=results_df,
    x="bias_trials",
    hue="optimiser",
    palette=[CustPalette.primary, CustPalette.secondary],
    linewidth=2.5,
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Bias Phase: Cumulative Completion Rate")
axes[1].set_xlabel("Trial Count (Retries)")
axes[1].set_ylabel("Percentage Completed")
axes[1].xaxis.set_major_locator(MultipleLocator(2))

# Formatting for both subplots
for ax in axes:
  ax.yaxis.set_major_formatter(PercentFormatter(1.0))
  ax.set_ylim(0, 1.05)  # Slightly above 100% for visibility
  ax.grid(True, linestyle="--", alpha=0.5)

# Build explicit legend handles using the palette
legend_elements = [
    Line2D([0], [0], color=CustPalette.primary, lw=2.5, label="llm"),
    Line2D([0], [0], color=CustPalette.secondary, lw=2.5, label="optuna"),
]

fig.legend(
    handles=legend_elements,
    title="Optimiser",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    frameon=False,
)

plt.tight_layout()
plt.show()

## Wall clock duration by Target Disparity

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(
    data=results_df,
    x='target_disp',
    y='wall_clock_sec',
    hue='optimiser',
    palette=[CustPalette.primary, CustPalette.secondary],
    medianprops={'linewidth': 2, 'color': 'white'},
    ax=ax,
)

ax.set_title('End-to-End Wall Clock Duration by Target Mode')
ax.set_xlabel('Disparity Target Mode')
ax.set_ylabel('Wall Clock Duration (seconds)')
ax.legend(
    title='Optimiser',
    loc="lower left", 
    bbox_to_anchor=(0, -0.3),
    ncol=2,
    frameon=False)

plt.tight_layout()
plt.show()

## LLM: Token consumption by target disparity

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Prepare LLM subset
llm_data = results_df[results_df['optimiser'] == 'llm'].copy()
llm_data['total_tokens'] = llm_data['input_tokens'] + llm_data['output_tokens']

fig, ax = plt.subplots(figsize=(7, 5))

sns.barplot(
  data=llm_data,
  hue='target_disp',
  y='total_tokens',
  palette=[CustPalette.primary_dark, CustPalette.primary, CustPalette.primary_med],
  estimator='mean',
  errorbar='sd',
  err_kws={'color':CustPalette.secondary},
  capsize=0.1,
  ax=ax,
)

# Label bar heights with exact mean token counts
for p in ax.patches:
  height = p.get_height()
  if height > 0:
    ax.annotate(
      f'{int(height):,}',
      (p.get_x() + p.get_width() / 2.0, height / 2.0),
      ha='center',
      va='center',
      color='white',
      fontweight='bold',
    )

ax.set_title('LLM Mean Token Consumption by Target Disparity Mode')
ax.set_xlabel('')
ax.set_ylabel('Total Tokens (Input + Output, ±1 SD)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(
  title="Target Disparity",
  loc="lower center", 
  bbox_to_anchor=(0.5, -0.18),
  ncol=3,
  frameon=False)

plt.tight_layout()
plt.show()

# Downstream Classifiers

In [ ]:
models = ['lr', 'rf', 'mlp']
metrics = ['auprc', 'recall_disp', 'ppv_disp']

tidy_rows = []
for idx, row in results_df.iterrows():
  for m in models:
    for metric in metrics:
      tidy_rows.append({
        'run_id': row['run_id'],
        'optimiser': row['optimiser'],
        'target_disp': row['target_disp'],
        'model': m.upper(),
        'metric': metric,
        'target_val': row[f'target_{metric}'],
        'probe_val': row[metric],
        'downstream_mean': row[f'downstream_{m}_{metric}_mean'],
        'downstream_std': row[f'downstream_{m}_{metric}_std'],
        'transfer_gap': row[f'downstream_{m}_{metric}_mean'] - row[metric],
        'target_gap': row[f'downstream_{m}_{metric}_mean']
        - row[f'target_{metric}'],
        'disp_tolerance': row['disp_tolerance'],
      })

downstream_df = pd.DataFrame(tidy_rows)
# downstream_df.head()

## Probe vs. Downstream Alignment (Transfer Fidelity)

In [ ]:

for row, optimiser in enumerate(['llm', 'optuna']):
  fig, axes = plt.subplots(1, 3, figsize=(16, 5))

  configs = [
    ('auprc', 'AUPRC: Probe vs. Downstream', axes[0]),
    ('recall_disp', 'Recall Disparity: Probe vs. Downstream', axes[1]),
    ('ppv_disp', 'PPV Disparity: Probe vs. Downstream', axes[2]),
  ]

  group = downstream_df[downstream_df['optimiser'] == optimiser]
  for metric_name, title, ax in configs:
    subset = group[group['metric'] == metric_name]
    
    if metric_name in ['recall_disp', 'ppv_disp']:
      subset = subset[subset['target_val'] > 0.0]

    sns.scatterplot(
      data=subset,
      x='probe_val',
      y='downstream_mean',
      hue='model',
      palette=[CustPalette.primary, CustPalette.secondary, CustPalette.tertiary],
      alpha=0.75,
      s=55,
      ax=ax,
    )

    # Identity line (y = x)
    min_val = min(subset['probe_val'].min(), subset['downstream_mean'].min())
    max_val = max(subset['probe_val'].max(), subset['downstream_mean'].max())
    lims = [min_val - 0.03, max_val + 0.03]
    ax.plot(lims, lims, color=CustPalette.grey_med, linestyle='--', label='y = x')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    ax.set_title(title)
    ax.set_xlabel(f'Pipeline Probe {metric_name.upper()}')
    ax.set_ylabel(f'Downstream Mean {metric_name.upper()}')
    ax.grid(True, linestyle='--', alpha=0.5)

    if ax != axes[2] and ax.get_legend() is not None:
      ax.get_legend().remove()

  axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
  plt.suptitle(f"{optimiser.upper()}", fontweight="bold")
  plt.tight_layout()
  plt.show()

## 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 5), sharey=True)

for idx, (m_name, label) in enumerate([
    ('auprc', 'AUPRC Gap'),
    ('recall_disp', 'Recall Disparity Gap'),
    ('ppv_disp', 'PPV Disparity Gap'),
]):
  sub = downstream_df[downstream_df['metric'] == m_name]
  # Only plot disparity if initially a target
  if m_name in ['recall_disp', 'ppv_disp']:
    sub = sub[sub['target_val'] > 0.0]

  sns.boxplot(
      data=sub,
      x='model',
      y='transfer_gap',
      hue='optimiser',
      palette=[CustPalette.primary, CustPalette.secondary],
      medianprops={'linewidth': 2, 'color': 'white'},
      width=0.4,
      gap=0.2,
      ax=axes[idx],
  )
  axes[idx].axhline(0, color=CustPalette.grey_dark, linestyle='--', linewidth=1)
  axes[idx].set_title(label)
  axes[idx].set_xlabel('Downstream Classifier')
  axes[idx].set_ylabel(
      'Transfer Gap (Downstream − Probe)' if idx == 0 else ''
  )
  axes[idx].grid(axis='y', linestyle='--', alpha=0.5)
  if idx > 0 and axes[idx].get_legend() is not None:
    axes[idx].get_legend().remove()

axes[0].legend(title='Optimiser', frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
def build_downstream_summary(df):
  # Exclude non-targeted 0.0 disparity runs
  targeted_df = df[
      (df['metric'] == 'auprc')
      | (
          (df['metric'] == 'recall_disp')
          & (df['target_disp'].isin(['recall', 'both']))
      )
      | ((df['metric'] == 'ppv_disp') & (df['target_disp'].isin(['ppv', 'both'])))
  ].copy()

  targeted_df['in_tol'] = (
      ((targeted_df['target_gap'].abs() <= targeted_df['disp_tolerance']) & (df['metric'] != 'auprc'))
      | 
      ((targeted_df['target_gap'] >= 0) & (df['metric'] == 'auprc'))
  )
  targeted_df['abs_gap'] = targeted_df['target_gap'].abs()

  summary = (
      targeted_df.groupby(['optimiser', 'model', 'metric'])[
          ['abs_gap', 'in_tol', 'downstream_std']
      ]
      .agg(
          mean_abs_gap=('abs_gap', 'mean'),
          tol_compliance_pct=('in_tol', lambda x: 100 * x.mean()),
          mean_model_std=('downstream_std', 'mean'),
      )
      .round(4)
      .reset_index()
  )

  return summary


downstream_summary_table = build_downstream_summary(downstream_df)
print(downstream_summary_table.to_markdown())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

metric_configs = [
    ('auprc', 'AUPRC Standard Deviation', None),
    ('recall_disp', 'Recall Disparity Std Dev', ['recall', 'both']),
    ('ppv_disp', 'PPV Disparity Std Dev', ['ppv', 'both']),
]

for idx, (m_name, title, targeted_modes) in enumerate(metric_configs):
  if targeted_modes is not None:
    sub = downstream_df[
        (downstream_df['metric'] == m_name)
        & (downstream_df['target_disp'].isin(targeted_modes))
    ]
  else:
    sub = downstream_df[downstream_df['metric'] == m_name]

  sns.barplot(
      data=sub,
      x='model',
      y='downstream_std',
      hue='optimiser',
      palette=[CustPalette.primary, CustPalette.secondary],
      estimator='mean',
      errorbar='se',
      capsize=0.1,
      ax=axes[idx],
  )

  axes[idx].set_title(title)
  axes[idx].set_xlabel('Downstream Model')
  axes[idx].set_ylabel('Mean Std Dev (±SE)' if idx == 0 else '')
  axes[idx].grid(axis='y', linestyle='--', alpha=0.5)

  if axes[idx].get_legend() is not None:
    axes[idx].get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles=handles,
    labels=labels,
    title='Optimiser',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    frameon=False,
)

plt.tight_layout()
plt.show()